# Ddri Station Clustering Baseline

- 범위: 강남구 따릉이 대여소
- 학습 데이터: 2023-01-01 ~ 2024-12-31
- 테스트 데이터: 2025-01-01 ~ 2025-12-31
- 1차 기준 대여소: 2023~2025 공통 169개 대여소


## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd().resolve().parents[1]
DATA_DIR = BASE_DIR / '3조 공유폴더'
OUTPUT_DATA_DIR = BASE_DIR / 'works' / 'clustering' / 'data'
OUTPUT_IMG_DIR = BASE_DIR / 'works' / 'clustering' / 'images'

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_IMG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')

## 2. Input Paths

In [ ]:
rental_2023_files = sorted((DATA_DIR / '2023 강남구 따릉이 이용정보').glob('*.csv'))
rental_2024_files = sorted((DATA_DIR / '2024 강남구 따릉이 이용정보').glob('*.csv'))
rental_2025_files = sorted((DATA_DIR / '2025 강남구 따릉이 이용정보').glob('*.csv'))

station_master_files = {
    2023: DATA_DIR / '강남구 대여소 정보 (2023~2025)' / '2023_강남구_대여소.csv',
    2024: DATA_DIR / '강남구 대여소 정보 (2023~2025)' / '2024_강남구_대여소.csv',
    2025: DATA_DIR / '강남구 대여소 정보 (2023~2025)' / '2025_강남구_대여소.csv',
}

len(rental_2023_files), len(rental_2024_files), len(rental_2025_files)

## 3. Build Common Station Master

In [ ]:
station_frames = {}
station_id_sets = {}

for year, path in station_master_files.items():
    df = pd.read_csv(path)
    df['대여소번호'] = pd.to_numeric(df['대여소번호'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['대여소번호']).copy()
    df['대여소번호'] = df['대여소번호'].astype(int)
    station_frames[year] = df
    station_id_sets[year] = set(df['대여소번호'])

common_station_ids = sorted(set.intersection(*station_id_sets.values()))
common_station_ids[:10], len(common_station_ids)

In [ ]:
station_master = (
    station_frames[2025]
    .loc[station_frames[2025]['대여소번호'].isin(common_station_ids), ['대여소번호', '대여소명', '자치구', '주소', '위도', '경도']]
    .drop_duplicates('대여소번호')
    .sort_values('대여소번호')
    .reset_index(drop=True)
)

station_master.head()

## 4. Load Rental Data

In [ ]:
RENTAL_COLS = ['대여일시', '대여 대여소번호', '반납일시', '반납대여소번호', '이용시간(분)', '이용거리(M)']

def load_rental_files(paths):
    frames = []
    for path in paths:
        df = pd.read_csv(path, usecols=RENTAL_COLS)
        df['대여일시'] = pd.to_datetime(df['대여일시'])
        df['대여 대여소번호'] = pd.to_numeric(df['대여 대여소번호'], errors='coerce').astype('Int64')
        df = df.dropna(subset=['대여일시', '대여 대여소번호']).copy()
        df['대여 대여소번호'] = df['대여 대여소번호'].astype(int)
        df = df[df['대여 대여소번호'].isin(common_station_ids)].copy()
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

train_df = load_rental_files(rental_2023_files + rental_2024_files)
test_df = load_rental_files(rental_2025_files)

train_df.shape, test_df.shape

## 5. Build Clustering Features

In [ ]:
def build_station_features(df):
    work = df.copy()
    work['date'] = work['대여일시'].dt.date
    work['hour'] = work['대여일시'].dt.hour
    work['weekday'] = work['대여일시'].dt.weekday
    work['is_weekend'] = work['weekday'] >= 5
    work['is_peak'] = work['hour'].isin([7, 8, 9, 18, 19, 20])
    work['is_night'] = work['hour'].isin([22, 23, 0, 1, 2, 3, 4, 5])

    daily = (
        work.groupby(['대여 대여소번호', 'date'])
        .size()
        .reset_index(name='daily_rental_count')
    )

    avg_rental = daily.groupby('대여 대여소번호')['daily_rental_count'].mean()
    rental_std = daily.groupby('대여 대여소번호')['daily_rental_count'].std().fillna(0)

    weekday_avg = (
        work.loc[~work['is_weekend']]
        .groupby(['대여 대여소번호', 'date'])
        .size()
        .groupby('대여 대여소번호')
        .mean()
    )

    weekend_avg = (
        work.loc[work['is_weekend']]
        .groupby(['대여 대여소번호', 'date'])
        .size()
        .groupby('대여 대여소번호')
        .mean()
    )

    total = work.groupby('대여 대여소번호').size()
    peak = work.loc[work['is_peak']].groupby('대여 대여소번호').size()
    night = work.loc[work['is_night']].groupby('대여 대여소번호').size()

    features = pd.DataFrame({
        'station_id': avg_rental.index,
        'avg_rental': avg_rental.values,
        'rental_std': rental_std.reindex(avg_rental.index).values,
        'weekday_avg': weekday_avg.reindex(avg_rental.index).fillna(0).values,
        'weekend_avg': weekend_avg.reindex(avg_rental.index).fillna(0).values,
        'peak_ratio': (peak / total).reindex(avg_rental.index).fillna(0).values,
        'night_ratio': (night / total).reindex(avg_rental.index).fillna(0).values,
    })
    features['weekday_weekend_gap'] = features['weekday_avg'] - features['weekend_avg']
    return features.sort_values('station_id').reset_index(drop=True)

train_features = build_station_features(train_df)
train_features.head()

## 6. Save Feature Table

In [ ]:
feature_path = OUTPUT_DATA_DIR / 'ddri_station_cluster_features_train_2023_2024.csv'
train_features.to_csv(feature_path, index=False)
feature_path

## 7. KMeans Search

In [ ]:
feature_cols = [
    'avg_rental',
    'rental_std',
    'weekday_avg',
    'weekend_avg',
    'peak_ratio',
    'night_ratio',
    'weekday_weekend_gap',
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_features[feature_cols])

search_rows = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X_scaled)
    search_rows.append({
        'k': k,
        'inertia': model.inertia_,
        'silhouette': silhouette_score(X_scaled, labels),
    })

k_search_df = pd.DataFrame(search_rows)
k_search_df

## 8. Elbow / Silhouette Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=k_search_df, x='k', y='inertia', marker='o', ax=axes[0])
sns.lineplot(data=k_search_df, x='k', y='silhouette', marker='o', ax=axes[1])
axes[0].set_title('Elbow Plot')
axes[1].set_title('Silhouette Score by K')
plt.tight_layout()
plt.show()

## 9. Final KMeans and PCA

In [ ]:
FINAL_K = 3

final_model = KMeans(n_clusters=FINAL_K, random_state=42, n_init=20)
train_features['cluster'] = final_model.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
pca_points = pca.fit_transform(X_scaled)
train_features['pca_1'] = pca_points[:, 0]
train_features['pca_2'] = pca_points[:, 1]

train_features.head()

## 10. Cluster Summary and Visuals

In [ ]:
cluster_summary = train_features.groupby('cluster')[feature_cols].mean().round(3)
cluster_summary

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=train_features, x='pca_1', y='pca_2', hue='cluster', palette='Set2')
plt.title('PCA Scatter by Cluster')
plt.tight_layout()
plt.show()